In [0]:
sales_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/sql_problems/default/my_volume/day03_sales.csv")

sales_df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

windowSpec = Window\
    .partitionBy(col("product_id")).orderBy(col("date")) \
    .rowsBetween(-2, Window.currentRow)

result_df = sales_df\
    .withColumn(
        "rolling_avg", round(avg(col("daily_sales")).over(windowSpec),2)
        )
display(result_df)



In [0]:
sales_df.createOrReplaceTempView("sales")

In [0]:
%sql
SELECT product_id, date, daily_sales,
ROUND(
    AVG(daily_sales) OVER(
        PARTITION BY product_id
        ORDER BY date 
        ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
    ),2
) as rolling_avg
FROM sales
ORDER BY product_id